# 02 Tracking: run a line in one code, several codes, and a real machine

Each code cell below is one of the seven scripts of `02_tracking/`, in order: the same code as the command line (`python 02_tracking/NN_name.py`), plus a first line setting `__file__`, which the scripts use to find `output/`. Change a cell to experiment; to change an example for good, edit its script and run `python sync_notebooks.py`, which copies the scripts into the notebooks. The text here adds the explanation; each script's docstring says what it shows, what it needs and what it checks.

08 to 12 use small lattices built in Python by `simple_lattice.py`: a drift and FODO cells, written out as a master lattice (YAML, sections, layouts and a `.def`) exactly like a machine in laura-lattices. Their answers are known from linear optics, so each code is checked against a transfer matrix worked out from first principles. 13 and 14 track JFEL.

Everything here is tier A (pip only). 11, 12 and 14 add elegant when the simcodes image is there (see the README). The whole notebook takes a minute or two, most of it in 14.

## 08 Track a beam through a drift

The smallest tracking run: two markers 2 m apart, a `.def` with one line, run in Ocelot. The `files:` entry names the code, the starting beam (`input: particle_definition:`) and the line's first and last elements (`output:`). After `track()` the run directory holds the input deck SIMBA wrote and a beam file at each marker, `<name>.openpmd.hdf5`.

In a drift, $\sigma_x^2(L) = \langle x^2\rangle + 2L\langle xx'\rangle + L^2\langle x'^2\rangle$, from the starting beam's own moments, so the check needs no Twiss approximation.

In [ ]:
__file__ = "08_track_drift.py"  # python and %run set this; a notebook cell does not
"""
08 - Track a beam through a drift
=================================

Shows: the smallest complete tracking run. A 2 m line with only two
markers (SIMBA fills the gap with a drift), a converging beam, one line in
the .def run by Ocelot, and the beam read back at the end.

The steps are the same for any machine:
  1. a master lattice folder: the LAURA YAML plus a .def (here written by
     simple_lattice.py; for JFEL they are in laura-lattices);
  2. Framework(master_lattice=..., directory=...) and loadSettings(".def");
  3. framework.track();
  4. read the beam files SIMBA wrote in `directory`, one per screen
     (marker) plus one at the end of each line, named after the element.

Run:    python 02_tracking/08_track_drift.py
Tier:   A
Needs:  SIMBA, LAURA and Ocelot.
Output: output/08_track_drift/lattice/ (the master lattice and START beam)
        and output/08_track_drift/run/ (Ocelot's input deck LINE.py, the
        beam END.openpmd.hdf5, and SIMBA's summaries).

Units: x in m; x' = px/pz (rad); emittance in m rad; momentum in eV/c.

The starting beam converges (alpha_x = alpha_y = 2 at beta = 5 m), so it
passes through a waist 2 m on, at the END marker, with beta* = 1/gamma = 1 m.

Check: a drift of length L maps x to x + L x', so the rms size at the end
is sigma^2(L) = <x^2> + 2L<x x'> + L^2 <x'^2>, with the moments taken from
the starting beam. The tracked rms sizes must match this to 1e-6
relative. Ocelot's drift is exact to second order, and the beam's
energy spread (1e-4) changes nothing at that level. N, charge, emittance
and mean momentum must be unchanged, to the same 1e-6.
"""

from pathlib import Path

import numpy as np
import simba.Framework as fw
import simba.Modules.Beams as rbf
from simple_lattice import make_beam, marker, sigma_matrix, write_master_lattice

OUT = Path(__file__).resolve().parents[1] / "output" / Path(__file__).stem
LATTICE = OUT / "lattice"
L = 2.0  # m

# The .def: one line, run by Ocelot, from START to END. Its beam is the file
# START.openpmd.hdf5 in the master lattice folder.
files = {
    "LINE": {
        "code": "ocelot",
        "input": {"particle_definition": "$master_lattice$START"},
        "output": {"start_element": "START", "end_element": "END"},
    }
}
settings = write_master_lattice(LATTICE, [marker("START", 0.0), marker("END", L)], files, "drift.def")
start = make_beam(LATTICE / "START.openpmd.hdf5", beta=(5.0, 5.0), alpha=(2.0, 2.0))

framework = fw.Framework(master_lattice=str(LATTICE), directory=str(OUT / "run"), clean=True, verbose=False)
framework.loadSettings(settings)
print(f"Line LINE: {framework['LINE'].code}, elements {list(framework['LINE'].elements)}")
framework.track()
end = rbf.load_file(str(OUT / "run" / "END.openpmd.hdf5"))

print(f"\n{'':12s} {'START':>12s} {'END':>12s} {'expected':>12s}")
problems = []
for plane in ("x", "y"):
    s = sigma_matrix(start, plane)
    expected = np.sqrt(s[0, 0] + 2 * L * s[0, 1] + L ** 2 * s[1, 1])
    got = np.std(getattr(end, plane).val)
    print(f"sigma_{plane} [um] {np.sqrt(s[0, 0]) * 1e6:12.3f} {got * 1e6:12.3f} {expected * 1e6:12.3f}")
    if abs(got / expected - 1) > 1e-6:
        problems.append(f"sigma_{plane}: {got:.6e} m, expected {expected:.6e} m")

# The same in Twiss terms: beta(L) = beta0 - 2 alpha0 L + gamma0 L^2.
P0, P1 = start.Particles.twiss, end.Particles.twiss
beta_expected = P0.beta_x.val - 2 * P0.alpha_x.val * L + (1 + P0.alpha_x.val ** 2) / P0.beta_x.val * L ** 2
print(f"beta_x [m]   {P0.beta_x.val:12.4f} {P1.beta_x.val:12.4f} {beta_expected:12.4f}"
      f"   (waist: alpha_x {P1.alpha_x.val:+.4f})")

quantities = {
    "N": lambda b: len(b.x),
    "charge [pC]": lambda b: b.Q * 1e12,
    "enx [um]": lambda b: b.Particles.emittance.enx.val * 1e6,
    "eny [um]": lambda b: b.Particles.emittance.eny.val * 1e6,
    "<cp> [MeV/c]": lambda b: np.mean(b.cp.val) / 1e6,
}
for name, fn in quantities.items():
    a, b = fn(start), fn(end)
    print(f"{name:12s} {a:12.6g} {b:12.6g}")
    if abs(b / a - 1) > 1e-6:
        problems.append(f"{name} changed: {a} -> {b}")

# --- Check -----------------------------------------------------------------
assert not problems, "\n".join(problems)
print("\nOK: rms sizes follow the drift's sigma matrix; N, charge, emittance and momentum unchanged")

## 09 Track through FODO cells

Three FODO cells with a screen halfway along each drift. SIMBA fills the gaps between elements with drifts, so the lattice only lists quadrupoles and markers. The beam is deliberately mismatched, so its size beats along the line.

At every screen, the beam's second-moment matrix must be $\Sigma = M\Sigma_0M^T$ with $M$ the thick-lens transfer matrix. The script then reads the whole run back with `load_directory` and plots the beam size and emittance along $z$. Pass `master_lattice=` to `load_directory` (ISSUES.md #22).

In [ ]:
__file__ = "09_track_fodo.py"  # python and %run set this; a notebook cell does not
"""
09 - Track through a FODO channel
=================================

Shows: tracking a beam through three FODO cells in Ocelot, with a screen
(marker) halfway along every drift. It reads the beam at the screens, loads
the whole run back with `load_directory`, and plots the beam size and
emittance along s.

Run:    python 02_tracking/09_track_fodo.py
Tier:   A
Needs:  SIMBA, LAURA and Ocelot.
Output: output/09_track_fodo/run/: one beam file per screen
        (SCR0A.openpmd.hdf5, ...) and END.openpmd.hdf5;
        output/09_track_fodo/envelope.png.

Units: x in m; x' = px/pz (rad); k1l in 1/m; emittance in m rad.

The lattice: 4 m cells, 0.2 m quadrupoles with k1l = +-0.6 /m (a phase
advance of 71 deg per cell). The beam starts with beta = 5 m and alpha = 0
in both planes. That isn't the cell's matched (periodic) beta, so the beam
size beats from cell to cell instead of repeating (10 shows the matched
beam).

load_directory(directory) reads a finished run back (the Twiss and beam
summaries, and with beams=True the beam at every screen), so you can look at
a run without tracking it again. On SIMBA main, give it the master lattice
too: without it, reading the run's settings.def fails when the .def names
its lattice files relative to the master lattice, as every laura-lattices
.def does (ISSUES.md #22). On main it also warns that `simcodes` and
`container_runtime` aren't set and prints "SimCodes not available". Both are
harmless here, because loading a run doesn't run a code (ISSUES.md #20).

Check: at every screen, the tracked beam's sigma matrix must match
M Sigma0 M^T, where Sigma0 is the starting beam's and M the thick-lens
transfer matrix from the start (simple_lattice.transfer_matrix), to 1e-4
relative. Ocelot also includes the second-order (chromatic) terms that the
matrix leaves out; they give a measured 2e-5, so the limit has 5x headroom.
The emittance must stay the same to 1e-4.
"""

from pathlib import Path

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import simba.Framework as fw
import simba.Modules.Beams as rbf
from simple_lattice import fodo, make_beam, sigma_matrix, transfer_matrix, write_master_lattice

OUT = Path(__file__).resolve().parents[1] / "output" / Path(__file__).stem
LATTICE = OUT / "lattice"
RUN = OUT / "run"

elements = fodo(cells=3)
files = {
    "LINE": {
        "code": "ocelot",
        "input": {"particle_definition": "$master_lattice$START"},
        "output": {"start_element": "START", "end_element": "END"},
    }
}
settings = write_master_lattice(LATTICE, elements, files, "fodo.def")
start = make_beam(LATTICE / "START.openpmd.hdf5", beta=(5.0, 5.0), alpha=(0.0, 0.0))

framework = fw.Framework(master_lattice=str(LATTICE), directory=str(RUN), clean=True, verbose=False)
framework.loadSettings(settings)
framework.track()

screens = [e for e in elements if e.hardware_class == "Marker" and e.name != "START"]
print(f"{'screen':8s} {'s [m]':>6s} {'sigma_x [um]':>13s} {'expected':>9s} {'sigma_y [um]':>13s} "
      f"{'expected':>9s} {'enx [um]':>9s}")
worst, problems = 0.0, []
for screen in screens:
    z = screen.physical.middle.z
    beam = rbf.load_file(str(RUN / f"{screen.name}.openpmd.hdf5"))
    row = f"{screen.name:8s} {z:6.2f}"
    for plane in ("x", "y"):
        m = transfer_matrix(elements, plane, 0.0, z)
        expected = m @ sigma_matrix(start, plane) @ m.T
        got = sigma_matrix(beam, plane)
        # each element of the matrix, on the scale of the diagonal (an off-diagonal one can be ~0)
        scale = np.sqrt(np.outer(np.diag(expected), np.diag(expected)))
        worst = max(worst, np.max(np.abs(got - expected) / scale))
        row += f" {np.sqrt(got[0, 0]) * 1e6:13.2f} {np.sqrt(expected[0, 0]) * 1e6:9.2f}"
    enx = beam.Particles.emittance.enx.val
    row += f" {enx * 1e6:9.5f}"
    print(row)
    if abs(enx / start.Particles.emittance.enx.val - 1) > 1e-4:
        problems.append(f"{screen.name}: enx {enx:.6e}, started at {start.Particles.emittance.enx.val:.6e}")
print(f"\nWorst difference from M Sigma0 M^T: {worst:.1e} (relative)")

# The whole run, read back from the directory. The plot's lines come from
# Ocelot's own optics output (Twiss_Summary.hdf5), one point per element.
run = fw.load_directory(str(RUN), master_lattice=str(LATTICE), beams=True)
print(f"load_directory: {len(run.twiss.z)} points along s, {len(run.beams.beams)} beam files")
run.plot(xkey="z", ykeys=["sigma_x", "sigma_y"], ykeys2=["enx", "eny"])
plt.savefig(OUT / "envelope.png", dpi=120, bbox_inches="tight")
print(f"Plot: {OUT / 'envelope.png'}")

# --- Check -----------------------------------------------------------------
assert worst < 1e-4, f"tracked sigma matrices differ from M Sigma0 M^T by {worst:.1e}"
assert not problems, "\n".join(problems)
print("\nOK: the beam at every screen follows the transfer matrices; emittance unchanged")

In [ ]:
from IPython.display import Image, display
display(Image(filename="../output/09_track_fodo/envelope.png"))

## 10 Periodic optics of a FODO cell

`tracking: {periodic: true}` asks the code for the matched optics of the line, from its one-turn (here one-cell) map, instead of carrying the beam's own Twiss parameters through it. `framework["LINE"].ring_parameters()` then gives the tune and the periodic $\beta$, $\alpha$ at the start.

Check: $\cos\mu = \mathrm{Tr}(M)/2$ and $\beta = M_{12}/\sin\mu$. The thin-lens FODO formulas are printed alongside; they are up to 7% off for these 0.2 m quadrupoles.

In [ ]:
__file__ = "10_twiss_without_particles.py"  # python and %run set this; a notebook cell does not
"""
10 - Periodic optics of a FODO cell
===================================

Shows: asking a code for the matched (periodic) optics of a line rather than
tracking a beam's own Twiss parameters through it. Set
`tracking: {periodic: true}` in the line's `files:` entry. After track(),
`framework["LINE"].ring_parameters()` gives the tune and the periodic Twiss
parameters at the start of the line, from the code's one-turn (here
one-cell) map. The Twiss parameters along the line are in the run's Twiss
summary, read with load_directory.

Ocelot, MAD-X, Xsuite, elegant, Bmad and ImpactX can do this; the other codes
warn and propagate the incoming beam's Twiss instead. A beam file is still
needed: here it only sets the reference momentum.

Run:    python 02_tracking/10_twiss_without_particles.py
Tier:   A
Needs:  SIMBA, LAURA and Ocelot.
Output: output/10_twiss_without_particles/twiss.png, the periodic beta
        functions along the cell.

Units: beta in m; tune in turns (1 = 360 deg of phase advance); k1l in 1/m.

The cell is 09's: 4 m long, starting and ending in the middle of a QF (two
0.1 m halves with k1l = 0.3 /m), a 0.2 m QD (k1l = -0.6 /m) in the middle.

Known noise on SIMBA main: reading Ocelot's optics after a periodic run
prints RuntimeWarnings ("invalid value encountered in divide") from
simba/Modules/Twiss/ocelot.py, because the beam-based columns are zero there
(ISSUES.md #24). They don't affect the lattice Twiss used here. SIMBA
also gives a NotClosedWarning ("its geometry does not close"): it checks
whether a periodic line closes on itself like a ring. A straight FODO cell
never does, and its periodic optics are still the matched ones.

Check: the cos(mu) = Tr(M)/2 and beta = M12/sin(mu) of the thick-lens cell
matrix M (simple_lattice.transfer_matrix) must give Ocelot's tune and
periodic beta to 1e-6, and every point of Ocelot's beta along the cell must
match the periodic Twiss carried through the matrices to 1e-6. The script
also prints the thin-lens FODO formulas,
sin(mu/2) = L/(4f), beta_max,min = L(1 +- sin(mu/2))/sin(mu), for
comparison. They are up to 7% off here (beta_y at a QF), because the
quadrupoles have length.
"""

from pathlib import Path

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import simba.Framework as fw
from simple_lattice import fodo, make_beam, transfer_matrix, write_master_lattice

OUT = Path(__file__).resolve().parents[1] / "output" / Path(__file__).stem
LATTICE = OUT / "lattice"
RUN = OUT / "run"
CELL, K1L = 4.0, 0.6

elements = fodo(cells=1, length=CELL, k1l=K1L)
files = {
    "LINE": {
        "code": "ocelot",
        "input": {"particle_definition": "$master_lattice$START"},
        "tracking": {"periodic": True},
        "output": {"start_element": "START", "end_element": "END"},
    }
}
settings = write_master_lattice(LATTICE, elements, files, "cell.def")
make_beam(LATTICE / "START.openpmd.hdf5", particles=1000)

framework = fw.Framework(master_lattice=str(LATTICE), directory=str(RUN), clean=True, verbose=False)
framework.loadSettings(settings)
framework.track()
ring = framework["LINE"].ring_parameters()
twiss = fw.load_directory(str(RUN), master_lattice=str(LATTICE)).twiss

# Thin lens: each quad a kick 1/f = |k1l|, the cell length L.
sin_half = CELL * K1L / 4
mu_thin = 2 * np.arcsin(sin_half)
thin = {"beta_max": CELL * (1 + sin_half) / np.sin(mu_thin), "beta_min": CELL * (1 - sin_half) / np.sin(mu_thin)}

print(f"{'':16s} {'Ocelot':>10s} {'matrix':>10s} {'thin lens':>10s}")
problems = []
for plane in ("x", "y"):
    m = transfer_matrix(elements, plane, 0.0, CELL)
    mu = np.arccos(np.trace(m) / 2)
    beta0 = m[0, 1] / np.sin(mu)
    alpha0 = (m[0, 0] - m[1, 1]) / (2 * np.sin(mu))
    beta_thin = thin["beta_max"] if plane == "x" else thin["beta_min"]  # the cell starts at a QF
    print(f"tune_{plane:11s} {ring[f'tune_{plane}']:10.6f} {mu / (2 * np.pi):10.6f} {mu_thin / (2 * np.pi):10.6f}")
    print(f"beta_{plane} at START {ring[f'beta_{plane}']:10.6f} {beta0:10.6f} {beta_thin:10.6f}")
    if abs(ring[f"tune_{plane}"] / (mu / (2 * np.pi)) - 1) > 1e-6:
        problems.append(f"tune_{plane}: Ocelot {ring[f'tune_{plane}']}, matrix {mu / (2 * np.pi)}")
    if abs(ring[f"beta_{plane}"] / beta0 - 1) > 1e-6:
        problems.append(f"beta_{plane}: Ocelot {ring[f'beta_{plane}']}, matrix {beta0}")

    # Carry the periodic Twiss along the cell: B(s) = M(0->s) B0 M(0->s)^T, B = [[beta, -alpha], [-alpha, gamma]].
    b0 = np.array([[beta0, -alpha0], [-alpha0, (1 + alpha0 ** 2) / beta0]])
    z = np.asarray(twiss.z.val)
    beta_code = np.asarray(getattr(twiss, f"beta_{plane}").val)
    beta_matrix = []
    for zi in z:
        # Ocelot gives the optics at element ends, so no point falls inside a quad.
        mi = transfer_matrix(elements, plane, 0.0, zi)
        beta_matrix.append((mi @ b0 @ mi.T)[0, 0])
    worst = np.max(np.abs(beta_code / np.array(beta_matrix) - 1))
    if worst > 1e-6:
        problems.append(f"beta_{plane} along the cell differs from the matrices by {worst:.1e}")
    plt.plot(z, beta_code, marker=".", label=f"beta_{plane} (Ocelot)")

print(f"\nbeta along the cell ({len(z)} points, at element ends): "
      f"beta_x {np.min(twiss.beta_x.val):.3f}..{np.max(twiss.beta_x.val):.3f} m, "
      f"beta_y {np.min(twiss.beta_y.val):.3f}..{np.max(twiss.beta_y.val):.3f} m")
plt.xlabel("s [m]")
plt.ylabel("periodic beta [m]")
plt.legend()
plt.savefig(OUT / "twiss.png", dpi=120, bbox_inches="tight")
print(f"Plot: {OUT / 'twiss.png'}")

# --- Check -----------------------------------------------------------------
assert not problems, "\n".join(problems)
print("\nOK: Ocelot's periodic tune and beta match the thick-lens cell matrix")

In [ ]:
display(Image(filename="../output/10_twiss_without_particles/twiss.png"))

## 11 Run the same line in different codes

`framework.change_Lattice_Code(line, code)` switches a line to another code without touching the lattice or the `.def`; `setSubDirectory` keeps each run separate. The same beam goes through 09's cells in Xsuite, Ocelot, Cheetah and MAD-X (and elegant), and the codes must agree particle by particle to $10^{-3}$ of the largest offset.

Two settings matter: MAD-X tracks a sliced (thin-lens) copy of the line, and Cheetah needs `cheetah_tracking_method: second_order` (in the `.def`'s `global:` block) to include chromatic focusing. The `XFAIL #23` line is a known SIMBA bug: `line.code` keeps the old name, although the tracking does switch.

In [ ]:
__file__ = "11_change_code.py"  # python and %run set this; a notebook cell does not
"""
11 - Run the same line in different codes
=========================================

Shows: `framework.change_Lattice_Code(line, code)`, which switches a line to
another tracking code without touching the lattice or the .def. The script
tracks the same beam through the same three FODO cells (09's) in Ocelot,
Cheetah, Xsuite and MAD-X. If a container runtime with the simcodes image is
there, it also runs elegant. Each run goes in its own directory
(setSubDirectory), and the script compares the codes particle by particle at
the end.

Run:    python 02_tracking/11_change_code.py
Tier:   A (elegant as well, if docker or apptainer and the simcodes image are there)
Needs:  SIMBA, LAURA, Ocelot, Cheetah, Xsuite and cpymad (MAD-X); the
        [simcodes] extra installs them all.
Output: output/11_change_code/<code>/: each code's input deck (LINE.py for
        Ocelot, LINE.madx, LINE.lte for elegant, ...) and END.openpmd.hdf5.

Units: x, y in m.

What differs between the codes:
- MAD-X tracks a thin-lens (sliced) copy of the line. SIMBA uses 4 slices per
  quadrupole by default and warns that the tunes can then differ from the
  thick optics by up to 0.01. Here that costs about 1e-4 in the particle
  positions. Set e.g. `tracking: {nslice_quadrupole: 16}` in the line's
  `files:` entry for more slices.
- Cheetah's default (`cheetah_tracking_method: auto`) tracks first-order
  maps unless that changes a beam size or emittance by more than 1%. First
  order leaves out chromatic focusing, which here moves particles by 1e-3 of
  the beam size (an energy spread of 1e-4 over 2/3 of a betatron period),
  so the script asks for `second_order` in the .def's `global:` block.

Check: every code must put every particle within 1e-3 of the beam's largest
offset from where Xsuite puts it, in x and y. That is the limit SIMBA's own
cross-code test uses (unit_tests/test_solenoid_and_alignment.py). The codes
must also keep every particle and the charge.
"""

import shutil
import subprocess
from pathlib import Path

import numpy as np
import simba.Framework as fw
import simba.Modules.Beams as rbf
from simple_lattice import fodo, make_beam, write_master_lattice

OUT = Path(__file__).resolve().parents[1] / "output" / Path(__file__).stem
LATTICE = OUT / "lattice"
IMAGE = "ghcr.io/astec-stfc/simcodes-docker:master"


def container_runtime():
    """
    docker if the simcodes image has been pulled, else apptainer if installed, else None.
    Docker comes first: with apptainer, SIMBA downloads its own image (several GB) on first use.
    """
    if shutil.which("docker") and subprocess.run(["docker", "image", "inspect", IMAGE],
                                                 capture_output=True).returncode == 0:
        return "docker"
    return "apptainer" if shutil.which("apptainer") else None


runtime = container_runtime()
codes = ["xsuite", "ocelot", "cheetah", "madx"] + (["elegant"] if runtime else [])
files = {
    "LINE": {
        "code": codes[0],
        "input": {"particle_definition": "$master_lattice$START"},
        "output": {"start_element": "START", "end_element": "END"},
    }
}
# Cheetah's first-order maps leave out chromatic focusing (see above), so ask
# for second order. `global:` in a .def holds per-code settings.
cheetah = {"CHEETAHsettings": {"cheetah_tracking_method": "second_order"}}
settings = write_master_lattice(LATTICE, fodo(cells=3), files, "fodo.def", **{"global": cheetah})
start = make_beam(LATTICE / "START.openpmd.hdf5", beta=(5.0, 5.0), alpha=(0.0, 0.0))

framework = fw.Framework(master_lattice=str(LATTICE), directory=str(OUT / codes[0]), clean=True,
                         verbose=False, container_runtime=runtime)
framework.loadSettings(settings)
ends = {}
stale = []
for code in codes:
    framework.change_Lattice_Code("LINE", code)
    framework.setSubDirectory(str(OUT / code))
    if framework["LINE"].code != code:
        stale.append(f"{code} (line.code {framework['LINE'].code!r})")
    framework.track()
    ends[code] = rbf.load_file(str(OUT / code / "END.openpmd.hdf5"))
# ISSUES.md #23: the tracking does switch code, but line.code (and the saved
# settings.def) keep the .def's code.
if stale:
    print(f"XFAIL #23: line.code not updated by change_Lattice_Code for {', '.join(stale)}")
else:
    print("XPASS #23: line.code follows change_Lattice_Code; remove this check and the note in ISSUES.md")
if not runtime:
    print("(elegant skipped: no container runtime with the simcodes image)")

ref = ends["xsuite"]
print(f"\nAt END, against xsuite (worst |difference| / largest |offset|):")
print(f"{'code':8s} {'N':>6s} {'sigma_x [um]':>13s} {'sigma_y [um]':>13s} {'x':>9s} {'y':>9s}")
problems = []
for code, beam in ends.items():
    row = f"{code:8s} {len(beam.x):6d} {np.std(beam.x.val) * 1e6:13.3f} {np.std(beam.y.val) * 1e6:13.3f}"
    if len(beam.x) != len(start.x) or not np.isclose(beam.Q, start.Q, rtol=1e-9, atol=0):
        problems.append(f"{code}: N {len(beam.x)}, charge {beam.Q} C")
        print(row)
        continue
    for plane in ("x", "y"):
        got, want = getattr(beam, plane).val, getattr(ref, plane).val
        worst = np.max(np.abs(got - want)) / np.max(np.abs(want))
        row += f" {worst:9.1e}"
        if worst > 1e-3:
            problems.append(f"{code}: {plane} differs from xsuite by {worst:.1e} of the largest offset")
    print(row)

# --- Check -----------------------------------------------------------------
assert not problems, "\n".join(problems)
print(f"\nOK: {', '.join(codes)} agree particle by particle to 1e-3")

## 12 Hand the beam from one code to the next

A `.def` with three lines in three codes. A line with no `input:` starts from the beam the previous line left at its `start_element`, and SIMBA converts the file format on the way. The chained run must match a single-code run of the whole channel, particle by particle, at each boundary.

In [ ]:
__file__ = "12_start_to_end_handoff.py"  # python and %run set this; a notebook cell does not
"""
12 - Hand the beam from one code to the next
============================================

Shows: a .def with several lines, each run by a different code, tracked in
one go. Each line runs from its start_element to its end_element. A line
with no `input:` starts from the beam the previous line left at its
start_element. SIMBA converts the beam between the codes' file formats
on the way.

Here three FODO cells (09's) are split at the screens into three lines:
  FIRST   START -> SCR0B   Ocelot
  SECOND  SCR0B -> SCR1B   elegant if a container runtime is there, else Cheetah
  THIRD   SCR1B -> END     Xsuite
Then the same beam goes through the whole channel in Xsuite alone.

Run:    python 02_tracking/12_start_to_end_handoff.py
Tier:   A (elegant as well, if docker or apptainer and the simcodes image are there)
Needs:  SIMBA, LAURA, Ocelot, Cheetah, Xsuite.
Output: output/12_start_to_end_handoff/chain/ (one input deck per line,
        e.g. FIRST.py, SECOND.lte, THIRD's Xsuite files, and the beam at
        each screen) and output/12_start_to_end_handoff/xsuite/ (the
        single-code run).

Units: x, y in m; charge in C.

Check: the beam must keep every particle and its charge across each hand-off,
and at the two boundaries and at END it must agree particle by particle with
the single-code run, to 1e-3 of the largest offset (the limit 11 uses). A
beam that jumps at a boundary (a lost particle, a sign or unit slip in a
conversion) would fail this.
"""

import shutil
import subprocess
from pathlib import Path

import numpy as np
import yaml
import simba.Framework as fw
import simba.Modules.Beams as rbf
from simple_lattice import fodo, make_beam, write_master_lattice

OUT = Path(__file__).resolve().parents[1] / "output" / Path(__file__).stem
LATTICE = OUT / "lattice"
IMAGE = "ghcr.io/astec-stfc/simcodes-docker:master"


def container_runtime():
    """docker if the simcodes image has been pulled, else apptainer if installed, else None (see 11)."""
    if shutil.which("docker") and subprocess.run(["docker", "image", "inspect", IMAGE],
                                                 capture_output=True).returncode == 0:
        return "docker"
    return "apptainer" if shutil.which("apptainer") else None


runtime = container_runtime()
middle = "elegant" if runtime else "cheetah"
files = {
    "FIRST": {
        "code": "ocelot",
        "input": {"particle_definition": "$master_lattice$START"},
        "output": {"start_element": "START", "end_element": "SCR0B"},
    },
    # no input: start from the beam FIRST wrote at SCR0B
    "SECOND": {"code": middle, "output": {"start_element": "SCR0B", "end_element": "SCR1B"}},
    "THIRD": {"code": "xsuite", "output": {"start_element": "SCR1B", "end_element": "END"}},
}
single = {
    "WHOLE": {
        "code": "xsuite",
        "input": {"particle_definition": "$master_lattice$START"},
        "output": {"start_element": "START", "end_element": "END"},
    }
}
cheetah = {"CHEETAHsettings": {"cheetah_tracking_method": "second_order"}}  # see 11
elements = fodo(cells=3)
write_master_lattice(LATTICE, elements, files, "chain.def", **{"global": cheetah})
start = make_beam(LATTICE / "START.openpmd.hdf5", beta=(5.0, 5.0), alpha=(0.0, 0.0))
# a second .def in the same folder, the same but for its files: block
deck = yaml.safe_load((LATTICE / "chain.def").read_text())
(LATTICE / "single.def").write_text(yaml.safe_dump({**deck, "files": single}, sort_keys=False))

results = {}
for name, settings in (("chain", "chain.def"), ("xsuite", "single.def")):
    framework = fw.Framework(master_lattice=str(LATTICE), directory=str(OUT / name), clean=True,
                             verbose=False, container_runtime=runtime)
    framework.loadSettings(settings)
    if name == "chain":
        for line in framework.lines:
            print(f"  {line:7s} {framework[line].code:8s} {framework[line].start} -> {framework[line].end}")
    framework.track()
    results[name] = {s: rbf.load_file(str(OUT / name / f"{s}.openpmd.hdf5")) for s in ("SCR0B", "SCR1B", "END")}

print(f"\n{'screen':7s} {'N':>6s} {'Q [pC]':>8s} {'sigma_x [um]':>13s} {'single code':>12s} {'worst x':>9s} {'worst y':>9s}")
problems = []
for screen, beam in results["chain"].items():
    ref = results["xsuite"][screen]
    row = f"{screen:7s} {len(beam.x):6d} {beam.Q * 1e12:8.3f} {np.std(beam.x.val) * 1e6:13.3f} {np.std(ref.x.val) * 1e6:12.3f}"
    if len(beam.x) != len(start.x) or not np.isclose(beam.Q, start.Q, rtol=1e-9, atol=0):
        problems.append(f"{screen}: N {len(beam.x)}, charge {beam.Q} C (started with {len(start.x)}, {start.Q} C)")
        print(row)
        continue
    for plane in ("x", "y"):
        got, want = getattr(beam, plane).val, getattr(ref, plane).val
        worst = np.max(np.abs(got - want)) / np.max(np.abs(want))
        row += f" {worst:9.1e}"
        if worst > 1e-3:
            problems.append(f"{screen}: {plane} differs from the single-code run by {worst:.1e}")
    print(row)

# --- Check -----------------------------------------------------------------
assert not problems, "\n".join(problems)
print(f"\nOK: the beam crossed ocelot -> {middle} -> xsuite with every particle and its charge, "
      "matching the single-code run")

## 13 A real machine: the JFEL linac

JFEL's `jfel_linac_codes.def` runs one line, Linac, from the saved injector beam (35 MeV, 250 pC) to the undulator entrance, 81 m on. The script switches it to Ocelot (or `--code ...`), turns off CSR, space charge and wakefields, and summarises the beam at the end of each section. The check is against the code-comparison table in JFEL's README.

To try another code from here: `parser.parse_args(["--code", "cheetah"])` in the cell below.

In [ ]:
__file__ = "13_jfel_linac.py"  # python and %run set this; a notebook cell does not
"""
13 - Track a real machine: the JFEL linac
=========================================

Shows: tracking JFEL (laura-lattices) from the saved injector beam (35 MeV,
250 pC, at the end of S02) through the linac, the X-band linearizer, the
bunch compressor and the main linac to the undulator entrance (81 m;
s in the table is measured from the start of the line), with
jfel_linac_codes.def. Its one line, Linac, runs in elegant by default;
the script switches it to Ocelot (tier A) unless you ask for another code.
It then summarises the beam at the end of each section, as JFEL's
Scripts/run_example.py does.

Run:    python 02_tracking/13_jfel_linac.py [--code ocelot|cheetah|madx|xsuite|elegant]
Tier:   A (elegant needs a container runtime)
Needs:  SIMBA, LAURA, laura-lattices and the chosen code.
Output: output/13_jfel_linac/run/: the input deck and a beam file at every
        screen and BPM (JFEL-<section>-....openpmd.hdf5).

Units: E in MeV (mean total momentum times c); energy spread in %;
emittance (normalised) in um; sizes in um; sigma_t in fs; current in A.

The .def rematches the injector beam to beta_x = 20 m, beta_y = 10 m
(`input: twiss:`) before tracking, so the first rows differ from the
injector file in size, not in energy or charge.

Collective effects are switched off here (csr_enable, lsc_enable and
wakefield_enable on the line), as for the reference table in the JFEL
README. Not every code models them, and 25 (phase 2) is about them.

Check: against the JFEL README's code-comparison table ("single-particle
optics"), at JFEL-FEL-SIM-MARK-01:
- mean energy within 0.2% of that code's value, the spread the README
  gives between codes;
- normalised emittances and beam sizes within 5%, the README's "a few %"
  between codes;
- no particle and no charge lost.
"""

import argparse
import os
from pathlib import Path

import numpy as np
import simba.Framework as fw
import simba.Modules.Beams as rbf
from laura_lattices import JFEL

OUT = Path(__file__).resolve().parents[1] / "output" / Path(__file__).stem
RUN = OUT / "run"
JFEL_DIR = os.path.dirname(JFEL.__file__)
END = "JFEL-FEL-SIM-MARK-01"

# JFEL README, "Code comparison", single-particle optics, at END:
# E [MeV], eps_nx, eps_ny, sigma_x, sigma_y [um]
README = {
    "elegant": (576.9, 0.762, 0.798, 49.3, 40.6),
    "ocelot": (576.8, 0.762, 0.812, 48.5, 40.1),
    "cheetah": (576.8, 0.762, 0.797, 48.4, 39.5),
    "madx": (576.7, 0.762, 0.798, 48.5, 39.5),
    "xsuite": (576.8, 0.756, 0.814, 49.1, 40.4),
}

parser = argparse.ArgumentParser()
parser.add_argument("--code", default="ocelot", choices=README)
code = parser.parse_args([]).code  # no command line here: options go in the []


def summary(beam):
    """E [MeV], spread [%], eps_nx, eps_ny, sigma_x, sigma_y [um], sigma_t [fs], peak current [A]."""
    P = beam.Particles
    cp = beam.cp.val
    t = beam.t.val
    counts, edges = np.histogram(t, bins=max(int(np.sqrt(len(t))), 10))
    q_per_particle = abs(beam.Q) / len(t)
    return (np.mean(cp) / 1e6, 100 * np.std(cp) / np.mean(cp), P.emittance.enx.val * 1e6,
            P.emittance.eny.val * 1e6, P.sigmas.sigma_x.val * 1e6, P.sigmas.sigma_y.val * 1e6,
            np.std(t) * 1e15, counts.max() * q_per_particle / (edges[1] - edges[0]))


framework = fw.Framework(master_lattice=JFEL_DIR, directory=str(RUN), clean=True, verbose=False,
                         container_runtime="docker" if code == "elegant" else None)
framework.loadSettings("Lattices/jfel_linac_codes.def")
framework.change_Lattice_Code("Linac", code)
line = framework["Linac"]
line.csr_enable = line.lsc_enable = line.wakefield_enable = False
# (line.code still names the .def's code after change_Lattice_Code on SIMBA main: ISSUES.md #23)
print(f"Linac: {code}, {line.start} -> {line.end}")
framework.track()

start = rbf.load_file(os.path.join(JFEL_DIR, "Data_Files", "jfel_injector_S02.openpmd.hdf5"))
# The beam at the last screen of each section, in order along the line. The
# accelerating sections (L02 ... L09, L4H) are left out: their screens sit
# at their entrance, so the next section's row shows what they did.
s = line.getSValues(as_dict=True)
last = {}
for name in s:
    if (RUN / f"{name}.openpmd.hdf5").exists() and not name.split("-")[1].startswith("L"):
        last[name.split("-")[1]] = name

row = "{:8.2f} {:7.3f} {:6.3f} {:6.3f} {:6.1f} {:6.1f} {:6.0f} {:6.0f}"
print(f"\n{'section':8s} {'s [m]':>6s} {'E [MeV]':>8s} {'dE [%]':>7s} {'enx':>6s} {'eny':>6s} "
      f"{'sig_x':>6s} {'sig_y':>6s} {'sig_t':>6s} {'I_pk':>6s}")
print(f"{'injector':8s} {0:6.1f} " + row.format(*summary(start)))
for section, name in last.items():
    beam = rbf.load_file(str(RUN / f"{name}.openpmd.hdf5"))
    print(f"{section:8s} {s[name]:6.1f} " + row.format(*summary(beam)))

end = rbf.load_file(str(RUN / f"{END}.openpmd.hdf5"))
E, _, enx, eny, sx, sy, _, _ = summary(end)
ref = README[code]
print(f"\nAt {END}, {code}: E {E:.1f} MeV (README {ref[0]}), eps_n {enx:.3f}/{eny:.3f} um "
      f"(README {ref[1]}/{ref[2]}), sigma {sx:.1f}/{sy:.1f} um (README {ref[3]}/{ref[4]})")

# --- Check -----------------------------------------------------------------
assert len(end.x) == len(start.x) and np.isclose(end.Q, start.Q, rtol=1e-9, atol=0), \
    f"lost particles or charge: {len(start.x)} -> {len(end.x)}, {start.Q} -> {end.Q} C"
assert abs(E / ref[0] - 1) < 0.002, f"E {E:.2f} MeV, README {ref[0]} MeV"
for label, got, want in zip(("eps_nx", "eps_ny", "sigma_x", "sigma_y"), (enx, eny, sx, sy), ref[1:]):
    assert abs(got / want - 1) < 0.05, f"{label} {got:.3f}, README {want}"
print("OK: energy, emittance and beam size at the undulator entrance agree with the JFEL README")

## 14 Compare codes on a real machine

13's line in Ocelot, Cheetah, MAD-X (and elegant), compared at every screen they share. Each pair's worst disagreement along the line is held to the ceiling SIMBA's own cross-code test uses. The codes are not expected to agree exactly: elegant compresses the bunch about 6% more (its RF model), and Ocelot grows $\varepsilon_{ny}$ by about 2% in the chicane.

`parser.parse_args(["--codes", "ocelot", "xsuite"])` in the cell below compares any set; pairs with no ceiling are printed but not checked.

In [ ]:
__file__ = "14_compare_codes.py"  # python and %run set this; a notebook cell does not
"""
14 - Compare codes on a real machine
====================================

Shows: the JFEL linac (13's line, from the saved injector beam to the
undulator entrance) tracked in several codes from the same beam, compared at
every screen they have in common, with a plot of the beam along s in each
code. It's a cut-down version of JFEL's Scripts/compare_codes.py.

Run:    python 02_tracking/14_compare_codes.py [--codes ocelot cheetah madx ...]
Tier:   A (adds elegant if a container runtime with the simcodes image is there)
Needs:  SIMBA, LAURA, laura-lattices, Ocelot, Cheetah, cpymad (MAD-X).
Output: output/14_compare_codes/<code>/ (each run) and
        output/14_compare_codes/compare.png.

Units: E in MeV; dE (rms energy spread) in %; emittance (normalised) and
sizes in um.

Collective effects are off in every code, as in 13: the codes don't all
model the same ones.

Where the codes differ (JFEL README, "Code comparison"):
- elegant (and Xsuite) compress the bunch ~6% more than the others: a
  different RF model gives a slightly different chirp into the chicane.
- Ocelot (and Xsuite, Bmad) grow eps_ny by ~2% in the chicane, from how the
  dipole edges are treated in y.
- sigma_y differs by several % around S11, where the beam has a ~25 um
  vertical waist and a few um is a large fraction.

Check: for each pair of codes, the largest disagreement anywhere along the
line in each quantity must be below the ceiling in SIMBA's own cross-code
test (unit_tests/test_jfel_code_comparison.py). That test runs the chicane
at 0.1 rad; this .def runs it at 0.13 rad, near full compression, so three
ceilings are re-measured here by the test's own rule (see TOLERANCES).
Relative quantities are in % of the pair's mean; dE is in percentage points.
Xsuite (--codes ... xsuite) is in neither table, so its pairs are printed
but not checked. At least 57 screens must be in common, the minimum SIMBA's
test asks for, so the comparison can't quietly shrink.
"""

import argparse
import itertools
import os
import shutil
import subprocess
from pathlib import Path

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import simba.Framework as fw
import simba.Modules.Beams as rbf
from laura_lattices import JFEL

OUT = Path(__file__).resolve().parents[1] / "output" / Path(__file__).stem
JFEL_DIR = os.path.dirname(JFEL.__file__)
IMAGE = "ghcr.io/astec-stfc/simcodes-docker:master"
SUFFIX = ".openpmd.hdf5"

# label: (function of the beam, compared relatively)
QUANTITIES = {
    "E": (lambda b: 1e-6 * np.mean(b.cp.val), True),
    "dE": (lambda b: 100 * np.std(b.cp.val) / np.mean(b.cp.val), False),
    "enx": (lambda b: 1e6 * b.Particles.emittance.enx.val, True),
    "eny": (lambda b: 1e6 * b.Particles.emittance.eny.val, True),
    "sx": (lambda b: 1e6 * b.Particles.sigmas.sigma_x.val, True),
    "sy": (lambda b: 1e6 * b.Particles.sigmas.sigma_y.val, True),
    "sz": (lambda b: 1e6 * b.Particles.sigmas.sigma_z.val, True),
}
# Ceilings (E, dE, enx, eny, sx, sy, sz): SIMBA's test_jfel_code_comparison.py
# (1.4x what it measured, with the chicane at 0.1 rad). Here the chicane is at
# the lattice's 0.13 rad, near full compression; three quantities measure more
# than the test's ceilings, so they get the test's rule, 1.4x the value measured
# here (2026-11-08, SIMBA main 26aed74):
#   madx/ocelot sy 3.96 and cheetah/ocelot sy 3.79: the ~25 um waist at S11;
#   elegant/ocelot sz 6.45: elegant compresses ~6% more (see above).
#                          E      dE     enx    eny    sx    sy    sz
TOLERANCES = {
    ("madx", "ocelot"):    (0.19, 0.087, 9.4,  5.6,  8.7,  5.6, 17.0),
    ("cheetah", "madx"):   (0.02, 0.01,  1.0,  2.0,  1.0,  1.0,  1.0),
    ("cheetah", "ocelot"): (0.19, 0.087, 9.5,  5.9,  8.7,  5.3, 17.0),
    ("cheetah", "elegant"): (0.16, 0.11, 11.0, 10.0, 11.0,  8.9, 20.0),
    ("elegant", "madx"):   (0.16, 0.11, 11.0,  9.6, 11.0,  9.1, 21.0),
    ("elegant", "ocelot"): (0.22, 0.026, 2.8,  6.7,  3.9,  7.8,  9.0),
}
MIN_SCREENS = 57


def docker_ready():
    return bool(shutil.which("docker")) and subprocess.run(
        ["docker", "image", "inspect", IMAGE], capture_output=True).returncode == 0


parser = argparse.ArgumentParser()
parser.add_argument("--codes", nargs="+", default=["ocelot", "cheetah", "madx"] + (["elegant"] if docker_ready() else []))
codes = parser.parse_args([]).codes  # no command line here: options go in the []


def track(code):
    """Track the Linac line in `code`; return {screen: {"s": s, quantity: value}}."""
    directory = OUT / code
    framework = fw.Framework(master_lattice=JFEL_DIR, directory=str(directory), clean=True, verbose=False,
                             container_runtime="docker" if code == "elegant" else None)
    framework.loadSettings("Lattices/jfel_linac_codes.def")
    framework.change_Lattice_Code("Linac", code)
    line = framework["Linac"]
    line.csr_enable = line.lsc_enable = line.wakefield_enable = False
    framework.track()
    stats = {}
    for f in sorted(directory.glob(f"*{SUFFIX}")):
        beam = rbf.load_file(str(f))
        row = {"s": float(np.atleast_1d(beam.s)[0])}
        row.update({k: float(fn(beam)) for k, (fn, _) in QUANTITIES.items()})
        stats[f.name[: -len(SUFFIX)]] = row
    return stats


def disagreement(a, b, relative):
    if not relative:
        return abs(a - b)
    return 100 * abs(a - b) / abs(0.5 * (a + b))


results = {code: track(code) for code in codes}
common = sorted(set.intersection(*(set(r) for r in results.values())), key=lambda n: results[codes[0]][n]["s"])
end = common[-1]
print(f"\n{len(common)} screens common to {', '.join(codes)}. At {end}:")
print(f"  {'':8s}" + "".join(f"{k:>9s}" for k in QUANTITIES))
for code in codes:
    print(f"  {code:8s}" + "".join(f"{results[code][end][k]:9.4g}" for k in QUANTITIES))

print("\nWorst disagreement along the line (% of the mean; dE in percentage points) / ceiling:")
print(f"  {'':18s}" + "".join(f"{k:>13s}" for k in QUANTITIES))
problems = []
for a, b in itertools.combinations(codes, 2):
    pair = (a, b) if (a, b) in TOLERANCES else (b, a)
    ceilings = TOLERANCES.get(pair)
    row = f"  {a + ' vs ' + b:18s}"
    for i, (k, (_, relative)) in enumerate(QUANTITIES.items()):
        worst = max(disagreement(results[a][n][k], results[b][n][k], relative) for n in common)
        row += f"{worst:8.3f}/{ceilings[i]:<4g}" if ceilings else f"{worst:8.3f}/-   "
        if ceilings and worst > ceilings[i]:
            problems.append(f"{a} vs {b}: {k} {worst:.3f} > {ceilings[i]}")
    print(row)

fig, axes = plt.subplots(len(QUANTITIES), 1, sharex=True, figsize=(8, 2 * len(QUANTITIES)))
for code in codes:
    s = [results[code][n]["s"] for n in common]
    for ax, k in zip(axes, QUANTITIES):
        ax.plot(s, [results[code][n][k] for n in common], marker=".", ms=3, label=code)
        ax.set_ylabel(k)
axes[0].legend(ncol=len(codes), fontsize="small")
axes[-1].set_xlabel("s [m]")
fig.tight_layout()
fig.savefig(OUT / "compare.png", dpi=120)
print(f"\nPlot: {OUT / 'compare.png'}")

# --- Check -----------------------------------------------------------------
assert len(common) >= MIN_SCREENS, f"only {len(common)} screens in common, expected >= {MIN_SCREENS}"
assert not problems, "Codes disagree by more than the ceilings:\n  " + "\n  ".join(problems)
print(f"\nOK: {', '.join(codes)} agree within the ceilings at all {len(common)} common screens")

In [ ]:
display(Image(filename="../output/14_compare_codes/compare.png"))